# laws_ingestion walkthrough (PoC)

Questo notebook mostra in modo compatto come avviene l'ingestion del corpus `leggi-html/` usando i metodi in `laws_ingestion/`.

Obiettivo: ottenere artifacts JSONL deterministici e "embedding-ready" (laws/articles/chunks/notes/edges) e verificare invarianti minimi (QA).


## 0) Setup percorsi

Nota: per velocita' usiamo `--scope benchmark` (22 leggi referenziate da `questions.csv`).


In [16]:
from pathlib import Path
import os
import json

def load_dotenv_simple(dotenv_path: Path) -> None:
    """Minimal .env loader (stdlib-only).
    Loads KEY=VALUE pairs into os.environ if not already set.
    """
    if not dotenv_path.exists():
        return
    for raw in dotenv_path.read_text(encoding="utf-8", errors="replace").splitlines():
        line = raw.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        k, v = line.split("=", 1)
        k = k.strip()
        v = v.strip().strip('"').strip("'")
        if k and k not in os.environ:
            os.environ[k] = v

def find_repo_root() -> Path:
    p = Path.cwd().resolve()
    for _ in range(6):
        if (p / "laws_ingestion").exists() and (p / "questions.csv").exists():
            return p
        p = p.parent
    raise RuntimeError("Cannot find repo root (expected laws_ingestion/ and questions.csv)")

ROOT = find_repo_root()
load_dotenv_simple(ROOT / ".env")

# HTML corpus path: env override, else try common defaults.
_html_env = os.environ.get("LAWS_INGESTION_HTML_DIR", "").strip()
if _html_env:
    HTML_DIR = Path(_html_env)
else:
    HTML_DIR = ROOT / "leggi-html"
    if not HTML_DIR.exists():
        alt = ROOT / "data" / "leggi-html"
        if alt.exists():
            HTML_DIR = alt

CSV_PATH = Path(os.environ.get("LAWS_INGESTION_CSV_PATH", str(ROOT / "questions.csv")))
OUT_DIR = Path(os.environ.get("LAWS_INGESTION_OUT_DIR", str(ROOT / "data" / "laws_ingestion_out")))

ROOT, HTML_DIR, HTML_DIR.exists(), CSV_PATH.exists(), OUT_DIR

(PosixPath('/Users/paolo.bonicco/Library/Mobile Documents/com~apple~CloudDocs/Università/Tesi/agentic-legal-rag-suite'),
 PosixPath('/Users/paolo.bonicco/Library/Mobile Documents/com~apple~CloudDocs/Università/Tesi/agentic-legal-rag-suite/data/leggi-html'),
 True,
 True,
 PosixPath('data/laws_ingestion_out'))

## 1) Benchmark: parsing + gold targets

Qui verifichiamo che `questions.csv` sia parsabile e che i riferimenti (legge+articolo) siano risolvibili nel corpus.

Riferimenti codice:
- `laws_ingestion/benchmark.py`
- `laws_ingestion/corpus_registry.py`


In [17]:
from laws_ingestion.corpus_registry import build_corpus_registry
from laws_ingestion.benchmark import iter_complete_questions, benchmark_summary, validate_gold_targets_exist

registry = build_corpus_registry(HTML_DIR)
questions = iter_complete_questions(CSV_PATH, corpus_registry=registry)
summary = benchmark_summary(questions)
print(json.dumps(summary, indent=2, sort_keys=True, ensure_ascii=False))

missing_count, missing_examples = validate_gold_targets_exist(questions, registry)
print("missing_gold_targets:", missing_count)
print("sample:", missing_examples[:3])

{
  "levels": {
    "L1": 25,
    "L2": 25,
    "L3": 25,
    "L4": 25
  },
  "questions": 100,
  "total_gold_refs": 106,
  "unique_law_article_referenced": 88,
  "unique_laws_referenced": 22
}
missing_gold_targets: 0
sample: []


## 2) Dataset determinismo: dataset_id

`dataset_id` e' un hash deterministico della snapshot di `leggi-html/` (contenuto dei file).

Riferimenti codice:
- `laws_ingestion/utils.py` (`compute_dataset_id`)


In [18]:
from laws_ingestion.utils import compute_dataset_id

dataset_id = compute_dataset_id(HTML_DIR.glob("*.html"))
print("dataset_id:", dataset_id)
print("html_files:", len(list(HTML_DIR.glob('*.html'))))

dataset_id: aa46ea3758c1de5596a8902b95e90cdfc6d08fbc2956086495a020680be22459
html_files: 3145


## 3) Ingestion di una singola legge (debug)

In questa sezione ispezioniamo l'output `law/articles/notes/edges` per una singola legge.

Riferimenti codice:
- `laws_ingestion/ingest.py`
- `laws_ingestion/debug.py`


In [19]:
from laws_ingestion.debug import debug_law

# scegli una legge "nota" del benchmark (cambia se vuoi)
law_id = "vda:lr:2000-01-25:5"
dbg = debug_law(html_dir=HTML_DIR, law_id=law_id, preview_articles=5, preview_chars=250)
print(json.dumps(dbg, indent=2, ensure_ascii=False))

{
  "law": {
    "law_id": "vda:lr:2000-01-25:5",
    "law_title": "Legge regionale 25 gennaio 2000, n. 5 - Testo vigente",
    "law_date": "2000-01-25",
    "law_number": 5,
    "is_abrogated": false,
    "abrogated_by_law_id": null,
    "source_file": "2577_LR-25-gennaio-2000-n5.html"
  },
  "counts": {
    "articles": 56,
    "notes": 63,
    "edges": 149
  },
  "articles_preview": [
    {
      "article_label_norm": "1",
      "article_heading": "(Principi fondamentali)",
      "anchor_name": "articolo_1__",
      "structure_path": "CAPO I",
      "text_preview": "1. La Regione, in armonia con lo Statuto speciale e le relative norme di attuazione, nonché in coerenza con l'autofinanziamento del servizio sanitario, applica i principi fondamentali di cui ai decreti legislativi 30 dicembre 1992, n. 502 (Riordino d"
    },
    {
      "article_label_norm": "2",
      "article_heading": "(Programmazione sanitaria regionale)",
      "anchor_name": "articolo_2__",
      "structure_path": "

## 4) Chunking embedding-ready su un articolo

Mostriamo come vengono prodotti i chunks con `text_for_embedding`.

Riferimenti codice:
- `laws_ingestion/chunking.py`
- `laws_ingestion/ingest.py` (`iter_chunks_for_articles`)


In [20]:
from laws_ingestion.ingest import ingest_law, iter_chunks_for_articles

law_file = registry.by_law_id[law_id]
law_rec, articles, notes, edges = ingest_law(law_file, registry)
print("articles:", len(articles), "notes:", len(notes), "edges:", len(edges))

chunks = list(iter_chunks_for_articles(law_rec, articles))
print("chunks:", len(chunks))
print("first_chunk_id:", chunks[0]["chunk_id"]) 
print("first_text_for_embedding:\n")
print(chunks[0]["text_for_embedding"][:800])

articles: 56 notes: 63 edges: 149
chunks: 60
first_chunk_id: vda:lr:2000-01-25:5#art:1#chunk:0
first_text_for_embedding:

[LR 2000-01-25 n.5] Legge regionale 25 gennaio 2000, n. 5 - Testo vigente | Art. 1 | CAPO I

1. La Regione, in armonia con lo Statuto speciale e le relative norme di attuazione, nonché in coerenza con l'autofinanziamento del servizio sanitario, applica i principi fondamentali di cui ai decreti legislativi 30 dicembre 1992, n. 502 (Riordino della disciplina in materia sanitaria, a norma dell'articolo 1 della legge 23 ottobre 1992, n. 421), 7 dicembre 1993, n. 517 (Modificazioni al d.lgs. 30 dicembre 1992, n. 502, recante riordino della disciplina in materia sanitaria, a norma dell'articolo 1 della L. 23 ottobre 1992, n. 421) e 19 giugno 1999, n. 229 (Norme per la razionalizzazione del Servizio sanitario nazionale, a norma dell'articolo 1 della legge 30 novembre 1998, n. 419), tramite: a) la


## 5) Ingestion end-to-end (scope=benchmark) -> artifacts JSONL

Qui eseguiamo la CLI per produrre artifacts in `OUT_DIR`.

Riferimenti codice:
- `laws_ingestion/__main__.py` (CLI)


In [21]:
import subprocess

cmd = ["python3", "-m", "laws_ingestion", "ingest", "--scope", "benchmark", "--html-dir", str(HTML_DIR), "--csv", str(CSV_PATH), "--out-dir", str(OUT_DIR)]
print(" ".join(cmd))
subprocess.run(cmd, check=True)

python3 -m laws_ingestion ingest --scope benchmark --html-dir /Users/paolo.bonicco/Library/Mobile Documents/com~apple~CloudDocs/Università/Tesi/agentic-legal-rag-suite/data/leggi-html --csv /Users/paolo.bonicco/Library/Mobile Documents/com~apple~CloudDocs/Università/Tesi/agentic-legal-rag-suite/questions.csv --out-dir data/laws_ingestion_out
{
  "dataset_id": "aa46ea3758c1de5596a8902b95e90cdfc6d08fbc2956086495a020680be22459",
  "schema_version": "v1",
  "parser_version": "0.1.4",
  "source_dir": "/Users/paolo.bonicco/Library/Mobile Documents/com~apple~CloudDocs/Università/Tesi/agentic-legal-rag-suite/data/leggi-html",
  "scope": "benchmark",
  "counts": {
    "laws": 22,
    "articles": 677,
    "chunks": 712,
    "notes": 556,
    "edges": 1394
  }
}


CompletedProcess(args=['python3', '-m', 'laws_ingestion', 'ingest', '--scope', 'benchmark', '--html-dir', '/Users/paolo.bonicco/Library/Mobile Documents/com~apple~CloudDocs/Università/Tesi/agentic-legal-rag-suite/data/leggi-html', '--csv', '/Users/paolo.bonicco/Library/Mobile Documents/com~apple~CloudDocs/Università/Tesi/agentic-legal-rag-suite/questions.csv', '--out-dir', 'data/laws_ingestion_out'], returncode=0)

## 6) QA artifacts (post-ingestion)

Controlli minimi: ID duplicati, testi vuoti, note sospette, % pseudo-articoli `unico`.

Riferimenti codice:
- `laws_ingestion/qa.py`


In [22]:
from laws_ingestion.qa import qa_artifacts

qa = qa_artifacts(artifacts_dir=OUT_DIR, schema_version="v1")
print(json.dumps(qa, indent=2, ensure_ascii=False, sort_keys=True))

{
  "artifacts_dir": "data/laws_ingestion_out",
  "counts": {
    "articles": 677,
    "chunks": 712,
    "edges": 1394,
    "laws": 22,
    "notes": 556
  },
  "dupes": {
    "article_id": 0,
    "chunk_id": 0,
    "edge_id": 0,
    "law_id": 0,
    "note_id": 0
  },
  "empty": {
    "article_text": 17,
    "chunk_text": 0,
    "note_text": 0
  },
  "pseudo_articles_unico": {
    "count": 0,
    "pct_of_articles": 0.0
  },
  "schema_version": "v1",
  "suspicious_notes": [
    {
      "chars": 6560,
      "law_id": "vda:lr:1998-04-06:11",
      "note_id": "vda:lr:1998-04-06:11#note:20",
      "note_number": "13"
    },
    {
      "chars": 16316,
      "law_id": "vda:lr:1998-04-06:11",
      "note_id": "vda:lr:1998-04-06:11#note:22",
      "note_number": "15"
    },
    {
      "chars": 5904,
      "law_id": "vda:lr:1998-04-06:11",
      "note_id": "vda:lr:1998-04-06:11#note:46",
      "note_number": "26"
    },
    {
      "chars": 13475,
      "law_id": "vda:lr:1998-04-06:11",
      

## (Opzionale) Ingestion full corpus

Sconsigliato in notebook se vuoi iterare velocemente; utile solo per run finali.


In [23]:
# cmd = ["python3", "-m", "laws_ingestion", "ingest", "--scope", "all", "--html-dir", str(HTML_DIR), "--out-dir", "/tmp/legal-rag-artifacts-all"]
# subprocess.run(cmd, check=True)
# print(json.dumps(qa_artifacts(artifacts_dir=Path("/tmp/legal-rag-artifacts-all"), schema_version="v1"), indent=2, ensure_ascii=False))
print("Skipped")

Skipped
